# Dental X-ray Analysis for Cavity Detection

This notebook runs the simplified implementation in `dental_xray_detection.py`.

It follows the class labs: Keras Sequential training, CNN/image augmentation, and transfer learning with freeze then fine-tune.

## 1. Setup & Imports

In [1]:
from pathlib import Path

from dental_xray_detection import (
    BEST_MODEL_PATH,
    OUTPUT_DIR,
    apply_training_augmentation,
    build_augmentation,
    build_model,
    combine_histories,
    ensure_dataset,
    evaluate_model,
    fine_tune,
    load_datasets,
    make_callbacks,
    print_summary,
    save_augmented_preview,
    save_combined_history_plot,
    set_reproducibility,
    train_head,
)

DATASET_PATH = Path("dataset")
BATCH_SIZE = 16
EPOCHS_HEAD = 5
EPOCHS_FINETUNE = 3
WEIGHTS = "imagenet"  # change to "none" for a quick offline smoke test

set_reproducibility()

## 2. Dataset Loading & Preprocessing

In [2]:
ensure_dataset(DATASET_PATH)
train_dataset, test_dataset, class_names, class_weight = load_datasets(DATASET_PATH, BATCH_SIZE)

Using TUFTS dataset: D:\8th term\NN\Cursor-LastDance\dataset\TUFTS
Label rule: image has any bbox row = cavity, otherwise no_cavity
Train images: 980 total | 335 cavity | 645 no_cavity
Test images: 20 total | 4 cavity | 16 no_cavity
Training batches: 62
Test batches:     2
Class weights:    {0: 0.7596899224806202, 1: 1.462686567164179}


## 3. Data Augmentation

In [3]:
data_augmentation = build_augmentation()
save_augmented_preview(train_dataset, data_augmentation)
augmented_train_dataset = apply_training_augmentation(train_dataset, data_augmentation)
print(f"Saved augmentation preview to {OUTPUT_DIR / 'augmented_samples.png'}")

Saved augmentation preview to outputs\augmented_samples.png


## 4. Model Architecture - Transfer Learning with MobileNetV2

In [4]:
dental_model, base_model = build_model(WEIGHTS)

Model: "dental_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,033 (8.93 MB)

 Trainable params: 82,049 (320.50 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Total parameters: 2,340,033
Trainable parameters (Phase 1): 82,049


## 5. Phase 1 Training: Classification Head

In [5]:
callbacks = make_callbacks()
history_phase1 = train_head(
    dental_model,
    augmented_train_dataset,
    test_dataset,
    callbacks,
    EPOCHS_HEAD,
    class_weight,
)


Phase 1: training the small classification head only.
Epoch 1/5

Epoch 1: val_accuracy improved from None to 0.75000, saving model to models\best_model.keras

Epoch 1: finished saving model to models\best_model.keras
62/62 - 9s - 152ms/step - accuracy: 0.5102 - loss: 0.8177 - val_accuracy: 0.7500 - val_loss: 0.6224
Epoch 2/5

Epoch 2: val_accuracy did not improve from 0.75000
62/62 - 3s - 52ms/step - accuracy: 0.5633 - loss: 0.6755 - val_accuracy: 0.7500 - val_loss: 0.6322
Epoch 3/5

Epoch 3: val_accuracy improved from 0.75000 to 0.80000, saving model to models\best_model.keras

Epoch 3: finished saving model to models\best_model.keras
62/62 - 3s - 52ms/step - accuracy: 0.5827 - loss: 0.6658 - val_accuracy: 0.8000 - val_loss: 0.6183
Epoch 4/5

Epoch 4: val_accuracy did not improve from 0.80000
62/62 - 3s - 44ms/step - accuracy: 0.6082 - loss: 0.6651 - val_accuracy: 0.7500 - val_loss: 0.6207
Epoch 5/5

Epoch 5: val_accuracy did not improve from 0.80000
62/62 - 3s - 48ms/step - accuracy

## 6. Phase 2 Fine-Tuning: Last 10 Layers

In [6]:
history_phase2 = fine_tune(
    dental_model,
    base_model,
    augmented_train_dataset,
    test_dataset,
    callbacks,
    EPOCHS_FINETUNE,
    class_weight,
)


Phase 2: fine-tuning only the last 10 MobileNetV2 layers.
MobileNetV2 layers: 154
Trainable MobileNetV2 layers: 10
Epoch 1/3

Epoch 1: val_accuracy did not improve from 0.80000
62/62 - 10s - 158ms/step - accuracy: 0.6704 - loss: 0.8009 - val_accuracy: 0.7000 - val_loss: 0.6178
Epoch 2/3

Epoch 2: val_accuracy did not improve from 0.80000
62/62 - 3s - 53ms/step - accuracy: 0.6776 - loss: 0.7519 - val_accuracy: 0.7000 - val_loss: 0.6300
Epoch 2: early stopping
Restoring model weights from the end of the best epoch: 1.
Phase 2 final train accuracy: 67.76%
Phase 2 final validation accuracy: 70.00%


## 7. Model Evaluation

In [7]:
combined_history = combine_histories(history_phase1, history_phase2)
phase1_epochs = len(history_phase1.history["accuracy"]) if history_phase1 else 0
save_combined_history_plot(combined_history, phase1_epochs)

test_loss, test_accuracy = evaluate_model(test_dataset, class_names)


Test Accuracy: 80.00% | Test Loss: 0.6183

Classification Report
              precision    recall  f1-score   support

   no_cavity       0.93      0.81      0.87        16
      cavity       0.50      0.75      0.60         4

    accuracy                           0.80        20
   macro avg       0.71      0.78      0.73        20
weighted avg       0.84      0.80      0.81        20



## 8. Results Summary

In [8]:
print_summary(combined_history, test_accuracy)
print(f"Saved model: {BEST_MODEL_PATH}")
print(f"Saved figures: {OUTPUT_DIR}")

  DENTAL X-RAY DETECTION - RESULTS SUMMARY
  Model:         MobileNetV2
  Image Size:    96x96 px
  Training:      CPU-friendly transfer learning
  Total Epochs:  7
  Best Val Acc:  80.00%
  Test Accuracy: 80.00%
Saved model: models\best_model.keras
Saved figures: outputs


## 9. Conclusion

The project uses the real TUFTS dental X-ray dataset and a lightweight transfer-learning workflow suitable for CPU training.